<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/Updates/Shipping_Regions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DO ITEMS SHIP FROM THE SAME PLACE?

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import joblib
from collections import defaultdict
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')

In [3]:
# Create a Product-Region Matrix
product_region = (
    df.groupby("Product Card Id")["Order Region"]
      .nunique()
      .reset_index(name="num_regions")
      .sort_values("num_regions", ascending=False)
)

print(product_region.head(20))

    Product Card Id  num_regions
15              191           23
36              403           23
44              627           23
35              365           23
37              502           23
68              957           23
12              134           22
13              135           21
53              825           21
10              116           20
49              818           20
39              565           20
61              893           20
25              276           19
38              564           19
55              835           19
64              906           19
20              235           19
4                44           19
21              249           19


In [39]:
order_features = (
    df.groupby("Order Id")
      .agg(
          unique_products=("Product Card Id", "nunique"),
          total_quantity=("Order Item Quantity", "sum"),
          total_sales=("Sales", "sum")
      )
      .reset_index()
)

order_features["complex_order"] = (
    order_features["unique_products"] >= 3
).astype(int)

# Create a product-region network

In [40]:
network = (
    df.groupby(["Product Card Id", "Order Region"])
      .agg(
          orders=("Order Id", "nunique"),
          quantity=("Order Item Quantity", "sum"),
          revenue=("Sales", "sum")
      )
      .reset_index()
)

# Find Product Statistics

In [41]:
product_stats = (
    network.groupby("Product Card Id")
           .agg(
               regions_served=("Order Region", "nunique"),
               total_orders=("orders", "sum"),
               total_quantity=("quantity", "sum"),
               total_revenue=("revenue", "sum")
           )
           .reset_index()
)

# Define Multi-Region Stock

In [42]:
region_threshold = product_stats["regions_served"].median()
order_threshold = product_stats["total_orders"].median()

product_stats["likely_multi_region_stocked"] = (
    (product_stats["regions_served"] >= region_threshold)
    &
    (product_stats["total_orders"] >= order_threshold)
).astype(int)

# Create a stock score

In [43]:
product_stats["stocking_score"] = (
    0.5 *
    (
        product_stats["regions_served"]
        / product_stats["regions_served"].max()
    )
    +
    0.5 *
    (
        product_stats["total_orders"]
        / product_stats["total_orders"].max()
    )
)


# Merge the network

In [44]:
network = network.merge(
    product_stats[
        [
            "Product Card Id",
            "regions_served",
            "stocking_score",
            "likely_multi_region_stocked"
        ]
    ],
    on="Product Card Id",
    how="left"
)

# Find edge weight

In [45]:
network["edge_weight"] = (
    0.4 *
    (network["orders"] / network["orders"].max())
    +
    0.3 *
    (network["quantity"] / network["quantity"].max())
    +
    0.3 *
    (network["revenue"] / network["revenue"].max())
)


# Find candidate shipping regions

In [46]:
candidate_regions = (
    network.sort_values(
        ["Product Card Id", "edge_weight"],
        ascending=[True, False]
    )
    .groupby("Product Card Id")
    .head(5)
)

# Save output

In [47]:
network.to_csv(
    "product_region_network.csv",
    index=False
)

candidate_regions.to_csv(
    "candidate_fulfillment_regions.csv",
    index=False
)

product_stats.to_csv(
    "product_stocking_scores.csv",
    index=False
)

print("Files created successfully")

Files created successfully


# RL State/Action/Reward

# RL Action